# SC4001 Phase 6: does DANN still fail on a representation that works?

Accelerator **GPU**. No internet. Code, tables, embeddings and k-mer offsets all come from
`isaacwjk21/sc4001-project`.

## The one hole left in the Phase 3 argument

Phase 3 concluded that adversarial domain adaptation does not help cold-target DTA, and ruled
out the configuration excuses: the adversary trained, compute was equal, lambda was swept with
damage growing monotonically, and a disabled-adversary control still lost ground.

One objection survived all of that. **Every DANN run was made on a protein representation that
cannot order unseen targets.** The character CNN's within-ligand concordance is 0.6789, barely
above chance ordering for this purpose, so "aligning feature marginals does not help" and
"there was nothing in those features worth aligning" predict the same result. The two cannot be
separated from Phase 3's runs alone.

Phase 5c closed that gap by accident. The k-mer residual hybrid reaches **within-ligand CI
0.7460** and a prediction spread of 0.795 of measured, the best of any arm in the project. So
the experiment is now available:

> **RQ.** When the protein representation demonstrably separates unseen targets, does adversarial
> domain alignment help, or does it still hurt?

## What each outcome means

* **DANN still hurts.** The Phase 3 conclusion is as strong as a three-seed study can make it.
  The failure is not about representation quality, and the write-up drops its remaining hedge.
  Combined with Zhao et al.'s lower bound and the fact that our unlabelled pool carries no
  labels at all, the claim becomes: marginal alignment is contraindicated here regardless of how
  good the features are.
* **DANN finally helps.** Better paper. The finding becomes a *condition*: alignment needs a
  representation that already separates the domains' content, and the Phase 3 negative was a
  statement about the CNN rather than about DANN. Phase 5 then reads as the prerequisite that
  the method always needed.

Both are reportable, which is why this is worth the last of the weekly allowance.

## Setup

Hybrid arm, DAVIS cold-target, ESM projection 96 plus the k-mer offset, 100 epochs with patience
equal to max-epochs. The adversary sees the protein branch's features, which is the Phase 3
configuration. Its unlabelled pool is the same 178 proteins (64 validation + 114 external
kinases), now embedded with ESM-2 rather than tokenised, and still contains **no test sequence** --
`load_target_domain` raises if it ever does.

Lambda 0.03 runs first because Phase 3 showed the adversary demonstrably trains there on DAVIS.
Lambda 0.1 follows if the wall-clock guard allows; it is the other value that trained.

## The validity gate, unchanged from Phase 3

A DANN run only counts as a measurement of the method if the adversary actually learned
something. The gate: `domain_loss` must sit at least 0.005 below ln 2 = 0.6931, or domain
accuracy must exceed 0.55. A run that fails the gate is a baseline with extra steps, and this
notebook reports it as such instead of folding it into the mean.

## Control

The hybrid without DANN, measured in `sc4001-phase5-kmer-residual` on the same seeds:
test MSE 0.3392 / 0.3451 / 0.3397 (mean 0.3413), test CI 0.8287 / 0.8293 / 0.8270 (mean 0.8284),
uncensored MSE 0.6808 / 0.7008 / 0.6990 (mean 0.6935).


In [ ]:
import json, os, shutil, subprocess, sys, time
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise SystemExit("No CUDA device. Set the accelerator to GPU T4 x2 or P100, not TPU or None.")
props = torch.cuda.get_device_properties(0)
print("torch", torch.__version__, "| device", props.name,
      "|", round(props.total_memory / 1e9, 1), "GB | count", torch.cuda.device_count())
KERNEL_STARTED = time.time()


In [ ]:
# Show what is actually mounted, so a layout surprise is visible rather than guessed at.
inputs = sorted(Path("/kaggle/input").rglob("*"))
print(len(inputs), "mounted paths; first 25:")
for path in inputs[:25]:
    print("   ", path)

# Locate the project root: the directory holding both src/train.py and data/processed/vocab.json.
# Kaggle strips the archive's top-level folder, so do not assume a "Project" path segment.
candidates = [p.parents[1] for p in Path("/kaggle/input").rglob("src/train.py")]
roots = [p for p in candidates if (p / "data" / "processed" / "vocab.json").exists()]
if not roots:
    raise SystemExit(
        "No project root found. Looked for src/train.py beside data/processed/vocab.json. "
        "Candidates with src/train.py but no vocab.json: " + str(candidates)
    )
source = sorted(roots)[0]

project = Path("/kaggle/working/sc4001/Project")
shutil.copytree(source, project, dirs_exist_ok=True)
env = dict(os.environ, PYTHONUNBUFFERED="1")
print("source :", source)
print("project:", project)
print("tables :", sorted(p.name for p in (project / "data" / "processed").iterdir()))

# This kernel needs the two modules added for Phase 5 and the extra split columns.
required = ["esm_embed.py", "kmer_baseline.py"]
present = {p.name for p in (project / "src").iterdir()}
missing = [name for name in required if name not in present]
if missing:
    raise SystemExit("Attached dataset predates Phase 5; missing src modules: " + str(missing)
                     + ". Push a new dataset version before running this kernel.")
print("phase 5 modules OK")


In [ ]:
# The frozen ESM-2 embeddings come from the attached dataset: they were computed once by
# sc4001-phase5-esm-coldtarget and uploaded, so nothing is downloaded or recomputed here and
# this kernel needs no internet. src/esm_embed.py is unchanged and still reproduces them.

# Verify every embedding file against its processed target table before training reads it.
import numpy as np
import pandas as pd

processed = project / "data" / "processed"
EXPECTED = {"davis": "davis_targets", "kiba": "kiba_targets", "external": "external_kinases"}
for source, table_stem in EXPECTED.items():
    path = processed / (source + "_esm.npz")
    if not path.exists():
        raise SystemExit("Missing " + str(path))
    with np.load(path, allow_pickle=False) as arrays:
        embeddings = arrays["prot_embeddings"]
    stem = processed / (table_stem + ".csv")
    table = pd.read_csv(stem if stem.exists() else stem.with_suffix(".csv.gz"), usecols=[0])
    if embeddings.shape != (len(table), 1280):
        raise SystemExit(source + " embeddings are " + str(embeddings.shape)
                         + ", expected " + str((len(table), 1280)))
    if not np.isfinite(embeddings).all():
        raise SystemExit(source + " embeddings contain a nonfinite value.")
    norms = np.linalg.norm(embeddings, axis=1)
    duplicates = len(table) - len(np.unique(embeddings, axis=0))
    print("%-9s %s  L2 norm min %.3f mean %.3f max %.3f  duplicate rows %d"
          % (source, embeddings.shape, norms.min(), norms.mean(), norms.max(), duplicates),
          flush=True)
    if duplicates:
        print("   NOTE: identical embeddings mean two targets are indistinguishable to the "
              "protein branch. DAVIS has 423 distinct sequences, so this should be 0.",
              flush=True)


In [ ]:
# DANN on the hybrid. Lambda 0.03 first; 0.1 only if the guard allows.
ROOT = "/kaggle/working/outputs"
GRID = [(0.03, seed) for seed in (4001, 4002, 4003)]
GRID += [(0.1, seed) for seed in (4001, 4002, 4003)]
EPOCHS = 100

# The hybrid baseline cost 1.7 min per DAVIS run. DANN adds a target-domain forward pass and a
# discriminator step per batch, so start from 3 min and let the first finished run correct it.
PRIOR_SECONDS = 180.0
BUDGET_SECONDS = 10 * 60

status, skipped = [], []
for lambda_max, seed in GRID:
    elapsed = time.time() - KERNEL_STARTED
    done = [row[3] for row in status if row[0] == lambda_max and row[2] == 0]
    projected = max(done) if done else PRIOR_SECONDS
    if elapsed + projected > BUDGET_SECONDS:
        skipped.append((lambda_max, seed, round(projected / 60, 1)))
        print("SKIPPING lambda " + str(lambda_max) + " seed" + str(seed) + ": projected "
              + format(projected / 60, ".1f") + " min would exceed the "
              + format(BUDGET_SECONDS / 60, ".0f") + " min guard ("
              + format(elapsed / 60, ".1f") + " min used)", flush=True)
        continue
    print()
    print("===== dann-on-hybrid davis lambda " + str(lambda_max) + " seed" + str(seed)
          + " =====", flush=True)
    t0 = time.time()
    run = subprocess.run(
        [sys.executable, "src/train_dann.py", "--dataset", "davis",
         "--split", "split_cold_target", "--seed", str(seed),
         "--protein-encoder", "esm", "--kmer-residual",
         "--lambda-max", str(lambda_max), "--discriminator-input", "protein",
         "--max-epochs", str(EPOCHS), "--patience", str(EPOCHS), "--resume"],
        cwd=project, env=dict(env, SC4001_OUTPUT_ROOT=ROOT), check=False,
    )
    wall = time.time() - t0
    status.append((lambda_max, seed, run.returncode, wall))
    print("----- rc=" + str(run.returncode) + " wall=" + format(wall / 60, ".1f") + " min -----",
          flush=True)

print()
print("Grid wall time", format((time.time() - KERNEL_STARTED) / 60, ".1f"), "min since start")
for lambda_max, seed, rc, wall in status:
    print("  lambda %-5s seed%d rc=%d %6.1f min" % (lambda_max, seed, rc, wall / 60))
failed = [row for row in status if row[2] != 0]
print("FAILED RUNS:", failed if failed else "none")
print("SKIPPED FOR BUDGET:", skipped if skipped else "none")


In [ ]:
# Paired against the hybrid without DANN, with the adversary-validity gate applied first.
import math
import statistics as st

LN2 = math.log(2.0)
GATE_MARGIN = 0.005
CONTROL = {"mse": [0.3392, 0.3451, 0.3397], "ci": [0.8287, 0.8293, 0.8270],
           "unc_mse": [0.6808, 0.7008, 0.6990]}
# For context: what DANN did to the CNN arm in Phase 3, on DAVIS.
PHASE3 = {"mse_delta": +0.0038, "note": "CNN arm, t(2)=+0.28, not significant"}

runs = {}
for path in sorted(Path(ROOT).glob("kmer-residual-phase3-esm/**/metrics.json")):
    payload = json.loads(path.read_text())
    saved = json.loads((path.parent / "config.json").read_text())
    runs.setdefault(saved["lambda_max"], []).append((payload, path.parent))

if not runs:
    print("No DANN runs found under", ROOT)

for lambda_max in sorted(runs):
    entries = sorted(runs[lambda_max], key=lambda row: row[0]["seed"])
    print()
    print("===== lambda %s, %d seed(s) =====" % (lambda_max, len(entries)))
    trained = []
    for payload, run_dir in entries:
        history = (run_dir / "history.csv").read_text().strip().splitlines()
        header = history[0].split(",")
        rows = [dict(zip(header, line.split(","))) for line in history[1:]]
        # Mean over the last 10 epochs, so one noisy epoch neither passes nor fails the gate.
        tail = rows[-10:]
        loss = st.mean(float(row["domain_loss"]) for row in tail)
        accuracy = st.mean(float(row["domain_accuracy"]) for row in tail)
        gate = (loss < LN2 - GATE_MARGIN) or (accuracy > 0.55)
        trained.append(gate)
        print("  seed%d  domain_loss %.4f (ln2 %.4f)  domain_acc %.4f  adversary trained: %s"
              % (payload["seed"], loss, LN2, accuracy, gate))
    if not all(trained):
        print("  At least one run's adversary did not train. Those runs are baselines with extra")
        print("  steps, not measurements of DANN, so the mean below is NOT a DANN result.")

    for label, key, control_key in (("test MSE", "mse", "mse"), ("test CI", "ci", "ci"),
                                    ("uncens MSE", "unc_mse", "unc_mse")):
        values = [p["test"]["all"][key] if key in ("mse", "ci")
                  else p["test"]["uncensored"]["mse"] for p, _ in entries]
        control = CONTROL[control_key][:len(values)]
        diffs = [a - b for a, b in zip(values, control)]
        mean, sd = st.mean(diffs), st.stdev(diffs) if len(diffs) > 1 else 0.0
        t = mean / (sd / len(diffs) ** 0.5) if sd > 0 else float("inf")
        signs = " ".join("+" if d > 0 else "-" for d in diffs)
        print("  %-11s dann %.4f  hybrid %.4f  delta %+.4f  t(2)=%+6.2f  signs %s"
              % (label, st.mean(values), st.mean(control), mean, t, signs))

    mse_delta = st.mean([p["test"]["all"]["mse"] for p, _ in entries]) - st.mean(CONTROL["mse"])
    print()
    if not all(trained):
        print("  Reading: inconclusive, the gate failed. Do not report this as a DANN result.")
    elif mse_delta > 0.002:
        print("  Reading: DANN still hurts on a representation that DOES order unseen targets.")
        print("  The Phase 3 conclusion is not an artefact of a weak protein encoder. On the CNN")
        print("  arm the same comparison gave %+.4f (%s)."
              % (PHASE3["mse_delta"], PHASE3["note"]))
    elif mse_delta < -0.002:
        print("  Reading: DANN HELPS once the representation separates unseen targets. The Phase 3")
        print("  negative was a statement about the CNN, not about the method, and a usable")
        print("  representation is the precondition the method always needed.")
    else:
        print("  Reading: neutral. DANN neither helps nor hurts here; at three seeds that is a")
        print("  null result, not evidence of no effect.")
